# Análisis de un ecommerce brasileño (Olist): retrasos, satisfacción y clientes

**Objetivo:** entender cómo afectan los retrasos en la entrega a la satisfacción de los clientes y a que vuelvan a comprar, e identificar dónde se concentran los retrasos (estados, categorías de producto y vendedores).

**Datos:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle), unos 100.000 pedidos de 2016 a 2018. Para reproducir el análisis, descarga los CSV y colócalos en `data/raw/`.

**Estructura del notebook:**
1. Carga de datos
2. Exploración de la tabla de pedidos
3. Cálculo de los retrasos
4. Retrasos y satisfacción
5. Retrasos por estado del cliente
6. Retrasos por categoría de producto
7. Retrasos por vendedor
8. Clientes que repiten
9. Segmentación de clientes (RFM)
10. Exportación de datos para Power BI
11. Conclusiones

## 1. Carga de datos

Cargamos las 9 tablas del dataset en un diccionario, para tenerlas todas a mano, y comprobamos su tamaño.

In [1]:
import pandas as pd

ruta = "data/raw/"

archivos = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

tablas = {}
for nombre, archivo in archivos.items():
    tablas[nombre] = pd.read_csv(ruta+archivo)
    print(f"{nombre}: {tablas[nombre].shape[0]} filas, {tablas[nombre].shape[1]} columnas")

orders: 99441 filas, 8 columnas
customers: 99441 filas, 5 columnas
order_items: 112650 filas, 7 columnas
payments: 103886 filas, 5 columnas
reviews: 99224 filas, 7 columnas
products: 32951 filas, 9 columnas
sellers: 3095 filas, 4 columnas
geolocation: 1000163 filas, 5 columnas
translation: 71 filas, 2 columnas


## 2. Exploración de la tabla de pedidos

Revisamos tipos de datos y valores nulos. Las fechas de entrega tienen nulos, así que comprobamos a qué estado de pedido corresponden.

In [2]:
orders = tablas["orders"]

print(orders.info())
print()
print("Valores nulos por columna:")
print(orders.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None

Valores nulos por columna:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_d

In [3]:
# Cuántos pedidos hay de cada estado
print(orders["order_status"].value_counts())
print()

# De los pedidos sin fecha de entrega, ¿qué estado tienen?
sin_entrega = orders[orders["order_delivered_customer_date"].isnull()]
print(sin_entrega["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


Casi todos los pedidos sin fecha de entrega no se han llegado a entregar (enviados, cancelados, no disponibles…), así que esos nulos tienen sentido y no son errores. Solo 8 pedidos marcados como *delivered* no tienen fecha; se excluyen del análisis de entregas.

Convertimos las columnas de fecha a formato fecha para poder calcular diferencias entre ellas.

In [4]:
columnas_fecha = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in columnas_fecha:
    orders[col] = pd.to_datetime(orders[col])

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


## 3. Cálculo de los retrasos

Nos quedamos con los pedidos entregados y calculamos los días que tardó cada uno y la diferencia con la fecha estimada de entrega (positivo = llegó tarde).

In [5]:
# 1. Nos quedamos solo con los pedidos entregados y con fecha de entrega
entregados = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notnull())
].copy()

# 2. Cuántos días tardó el pedido desde la compra
entregados["dias_entrega"] = (
    entregados["order_delivered_customer_date"] - entregados["order_purchase_timestamp"]
).dt.days

# 3. Diferencia con la fecha estimada (positivo = llegó tarde)
entregados["dias_retraso"] = (
    entregados["order_delivered_customer_date"].dt.normalize()
    - entregados["order_estimated_delivery_date"]
).dt.days

# 4. Marcamos si llegó tarde o no
entregados["retrasado"] = entregados["dias_retraso"] > 0

print(f"Pedidos analizados: {len(entregados)}")
print()
print(entregados[["dias_entrega", "dias_retraso"]].describe())
print()
print("Porcentaje de pedidos retrasados:")
print(entregados["retrasado"].value_counts(normalize=True) * 100)

Pedidos analizados: 96470

       dias_entrega  dias_retraso
count  96470.000000  96470.000000
mean      12.093604    -11.875889
std        9.551380     10.182105
min        0.000000   -147.000000
25%        6.000000    -17.000000
50%       10.000000    -12.000000
75%       15.000000     -7.000000
max      209.000000    188.000000

Porcentaje de pedidos retrasados:
retrasado
False    93.22691
True      6.77309
Name: proportion, dtype: float64


El **6,8 %** de los pedidos entregados llegó después de la fecha estimada. De media, un pedido tarda unos 12 días en llegar.

## 4. Retrasos y satisfacción

Unimos los pedidos con sus reseñas. Antes comprobamos si hay pedidos con más de una reseña, porque en ese caso el *merge* duplicaría pedidos y falsearía las medias.

In [6]:
reviews = tablas["reviews"]

# Comprobación: ¿hay pedidos con más de una reseña?
print("Pedidos con más de una reseña:", reviews["order_id"].duplicated().sum())
print()

# Unimos pedidos entregados con su puntuación
pedidos_reviews = entregados.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

print(f"Pedidos con reseña: {len(pedidos_reviews)}")
print()
print("Puntuación media según si llegó tarde o no:")
print(pedidos_reviews.groupby("retrasado")["review_score"].mean())

Pedidos con más de una reseña: 551

Pedidos con reseña: 96353

Puntuación media según si llegó tarde o no:
retrasado
False    4.289980
True     2.271025
Name: review_score, dtype: float64


Hay 551 pedidos con más de una reseña. Nos quedamos con la más reciente de cada pedido y repetimos la unión.

In [7]:
# Ordenamos por fecha de la reseña y nos quedamos con la última de cada pedido
reviews_unicas = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
)

print("Reseñas antes:", len(reviews), "| después:", len(reviews_unicas))
print()

# Repetimos la unión con las reseñas sin duplicados
pedidos_reviews = entregados.merge(
    reviews_unicas[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

print(f"Pedidos con reseña: {len(pedidos_reviews)}")
print()
print("Puntuación media según si llegó tarde o no:")
print(pedidos_reviews.groupby("retrasado")["review_score"].mean())
print()
print("Distribución de puntuaciones (%):")
print(pd.crosstab(pedidos_reviews["retrasado"], pedidos_reviews["review_score"], normalize="index") * 100)

Reseñas antes: 99224 | después: 98673

Pedidos con reseña: 95824

Puntuación media según si llegó tarde o no:
retrasado
False    4.290274
True     2.270804
Name: review_score, dtype: float64

Distribución de puntuaciones (%):
review_score          1         2          3          4          5
retrasado                                                         
False          6.618740  2.649732   8.075534  20.397348  62.258645
True          53.784673  8.619339  10.876038  10.170820  16.549130


Los pedidos que llegan tarde tienen una nota media de **2,27**, frente a **4,29** de los que llegan a tiempo. Más de la mitad de los pedidos retrasados (53,8 %) recibe 1 estrella.

## 5. Retrasos por estado del cliente

In [8]:
customers = tablas["customers"]

# Añadimos el estado del cliente a cada pedido
pedidos_estado = pedidos_reviews.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

# Resumen por estado
resumen_estado = pedidos_estado.groupby("customer_state").agg(
    pedidos=("order_id", "count"),
    dias_entrega_medio=("dias_entrega", "mean"),
    pct_retrasados=("retrasado", "mean"),
    nota_media=("review_score", "mean"),
)

resumen_estado["pct_retrasados"] = resumen_estado["pct_retrasados"] * 100

print(resumen_estado.sort_values("pct_retrasados", ascending=False).round(2))

                pedidos  dias_entrega_medio  pct_retrasados  nota_media
customer_state                                                         
AL                  394               23.96           20.81        3.85
MA                  712               20.99           17.13        3.83
SE                  334               20.95           14.97        3.91
PI                  471               19.03           13.80        3.99
CE                 1273               20.76           13.67        3.94
RR                   41               28.98           12.20        3.90
RJ                12211               14.78           11.92        3.97
BA                 3229               18.78           11.89        3.93
PA                  933               23.18           10.72        3.91
ES                 1969               15.15           10.41        4.08
PB                  512               19.77           10.16        4.08
TO                  273               17.25            9.89     

Los estados del norte y nordeste (AL, MA, SE, PI, CE…) concentran los porcentajes de retraso más altos. São Paulo (SP), el estado con más pedidos, está entre los que menos retrasos tienen (4,4 %).

## 6. Retrasos por categoría de producto

Un pedido puede incluir productos de varias categorías, así que cada pedido cuenta una vez por categoría. Solo se muestran las categorías con al menos 500 pedidos, para que los porcentajes sean fiables.

In [9]:
order_items = tablas["order_items"]
products = tablas["products"]
translation = tablas["translation"]

# 1. Categoría de cada producto, en inglés
productos_cat = products[["product_id", "product_category_name"]].merge(
    translation, on="product_category_name", how="left"
)

# 2. Qué categorías tiene cada pedido (cada pedido cuenta una vez por categoría)
pedidos_cat = (
    order_items[["order_id", "product_id"]]
    .merge(productos_cat, on="product_id", how="left")
    [["order_id", "product_category_name_english"]]
    .drop_duplicates()
)

# 3. Añadimos los datos de retraso y reseña
pedidos_cat = pedidos_cat.merge(
    pedidos_reviews[["order_id", "dias_entrega", "retrasado", "review_score"]],
    on="order_id",
    how="inner"
)

# 4. Resumen por categoría (solo las que tienen al menos 500 pedidos)
resumen_cat = pedidos_cat.groupby("product_category_name_english").agg(
    pedidos=("order_id", "count"),
    dias_entrega_medio=("dias_entrega", "mean"),
    pct_retrasados=("retrasado", "mean"),
    nota_media=("review_score", "mean"),
)
resumen_cat["pct_retrasados"] = resumen_cat["pct_retrasados"] * 100
resumen_cat = resumen_cat[resumen_cat["pedidos"] >= 500]

print(f"Categorías con al menos 500 pedidos: {len(resumen_cat)}")
print()
print(resumen_cat.sort_values("pct_retrasados", ascending=False).round(2))

Categorías con al menos 500 pedidos: 25

                                 pedidos  dias_entrega_medio  pct_retrasados  \
product_category_name_english                                                  
office_furniture                    1244               20.12            7.96   
baby                                2787               12.12            7.89   
electronics                         2498               12.43            7.61   
musical_instruments                  605               12.60            7.60   
health_beauty                       8601               11.58            7.36   
bed_bath_table                      9177               12.47            7.28   
watches_gifts                       5452               12.28            7.25   
auto                                3791               11.79            7.20   
construction_tools_construction      731               10.33            7.11   
telephony                           4069               12.47            7.05   

Las diferencias entre categorías son pequeñas (entre el 4 % y el 8 % de retrasos). Destaca *office_furniture*, con más días de entrega (20 de media) y la nota más baja (3,64).

## 7. Retrasos por vendedor

Solo se analizan vendedores con al menos 50 pedidos: con pocos pedidos, un único retraso dispara el porcentaje.

In [10]:
sellers = tablas["sellers"]

# 1. Qué vendedores participan en cada pedido (cada pareja pedido-vendedor una vez)
pedidos_vend = (
    order_items[["order_id", "seller_id"]]
    .drop_duplicates()
    .merge(
        pedidos_reviews[["order_id", "retrasado", "review_score"]],
        on="order_id",
        how="inner"
    )
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
)

# 2. Resumen por vendedor
resumen_vend = pedidos_vend.groupby("seller_id").agg(
    estado=("seller_state", "first"),
    pedidos=("order_id", "count"),
    pct_retrasados=("retrasado", "mean"),
    nota_media=("review_score", "mean"),
)
resumen_vend["pct_retrasados"] = resumen_vend["pct_retrasados"] * 100

# 3. Solo vendedores con volumen suficiente
vend_grandes = resumen_vend[resumen_vend["pedidos"] >= 50]

print(f"Vendedores en total: {len(resumen_vend)}")
print(f"Vendedores con al menos 50 pedidos: {len(vend_grandes)}")
print()
print("Distribución del % de retrasos entre vendedores grandes:")
print(vend_grandes["pct_retrasados"].describe().round(2))
print()
print("Los 10 vendedores con más retrasos:")
print(vend_grandes.sort_values("pct_retrasados", ascending=False).head(10).round(2))
print()
print("Estado de los vendedores (% de pedidos):")
print((pedidos_vend["seller_state"].value_counts(normalize=True) * 100).head(5).round(2))

Vendedores en total: 2965
Vendedores con al menos 50 pedidos: 418

Distribución del % de retrasos entre vendedores grandes:
count    418.00
mean       6.35
std        4.24
min        0.00
25%        3.42
50%        5.63
75%        8.66
max       28.99
Name: pct_retrasados, dtype: float64

Los 10 vendedores con más retrasos:
                                 estado  pedidos  pct_retrasados  nota_media
seller_id                                                                   
54965bbe3e4f07ae045b90b0b8541f52     PR       69           28.99        3.14
beadbee30901a7f61d031b6b686095ad     SP       64           23.44        3.91
a49928bcdf77c55c6d6e05e09a9b4ca5     SP       96           21.88        3.03
712e6ed8aa4aa1fa65dab41fed5737e4     SC       76           19.74        3.39
06a2c3af7b3aee5d69171b0e14f0ee87     MA      385           18.44        4.01
ea566164622c6b439516ab18062c42cd     SP       50           18.00        3.78
002100f778ceb8431b7a1020ff7ab48f     SP       50          

## 8. Clientes que repiten

En este dataset `customer_id` cambia en cada pedido. Para identificar a la misma persona usamos `customer_unique_id`.

In [11]:
# Unimos todos los pedidos (no solo los entregados) con el identificador real del cliente
pedidos_clientes = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# Cuántos pedidos ha hecho cada persona
compras_por_cliente = pedidos_clientes.groupby("customer_unique_id")["order_id"].count()

print(f"Clientes únicos: {len(compras_por_cliente)}")
print()
print("Número de compras por cliente:")
print(compras_por_cliente.value_counts().sort_index())
print()
repiten = (compras_por_cliente > 1).mean() * 100
print(f"Porcentaje de clientes que repiten: {repiten:.2f} %")

Clientes únicos: 96096

Número de compras por cliente:
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

Porcentaje de clientes que repiten: 3.12 %


Solo el **3,1 %** de los clientes vuelve a comprar. ¿Influye que su primera compra llegara tarde?

In [12]:
# 1. Primera compra de cada cliente
primeras = (
    pedidos_clientes.sort_values("order_purchase_timestamp")
    .drop_duplicates(subset="customer_unique_id", keep="first")
    [["customer_unique_id", "order_id", "order_purchase_timestamp"]]
)

# 2. Marcamos si el cliente volvió a comprar
primeras["repite"] = primeras["customer_unique_id"].map(compras_por_cliente) > 1

# 3. Añadimos si esa primera compra llegó tarde y qué nota recibió
primeras = primeras.merge(
    pedidos_reviews[["order_id", "retrasado", "review_score"]],
    on="order_id",
    how="inner"
)

print(f"Clientes analizados: {len(primeras)}")
print()
print("% que repite según si su primera compra llegó tarde:")
print((primeras.groupby("retrasado")["repite"].mean() * 100).round(2))
print()
print("% que repite según la nota que dio en su primera compra:")
print((primeras.groupby("review_score")["repite"].mean() * 100).round(2))

Clientes analizados: 92632

% que repite según si su primera compra llegó tarde:
retrasado
False    3.12
True     2.55
Name: repite, dtype: float64

% que repite según la nota que dio en su primera compra:
review_score
1    3.11
2    2.93
3    3.02
4    2.81
5    3.19
Name: repite, dtype: float64


Comprobamos con un test de chi-cuadrado si la diferencia entre clientes con y sin retraso en su primera compra es estadísticamente significativa.

In [13]:
from scipy.stats import chi2_contingency

# Tabla de recuentos: retrasado (sí/no) frente a repite (sí/no)
tabla = pd.crosstab(primeras["retrasado"], primeras["repite"])
print(tabla)
print()

chi2, p_valor, gl, esperados = chi2_contingency(tabla)
print(f"p-valor: {p_valor:.4f}")
print("Diferencia significativa" if p_valor < 0.05 else "Diferencia NO significativa (podría ser casualidad)")

repite     False  True 
retrasado              
False      83736   2697
True        6041    158

p-valor: 0.0133
Diferencia significativa


p = 0,013 < 0,05: la diferencia es **significativa**. Los clientes cuya primera compra llegó tarde repiten menos, aunque la diferencia es pequeña en términos absolutos porque en general muy pocos clientes repiten.

## 9. Segmentación de clientes (RFM)

Calculamos para cada cliente la **R**ecencia (días desde su última compra), la **F**recuencia (número de pedidos) y el valor **M**onetario (gasto total). Excluimos pedidos cancelados y no disponibles.

In [14]:
payments = tablas["payments"]

# 1. Importe total de cada pedido (un pedido puede tener varios pagos)
importe_pedido = payments.groupby("order_id")["payment_value"].sum().rename("importe")

# 2. Pedidos válidos: quitamos cancelados y no disponibles
validos = pedidos_clientes[
    ~pedidos_clientes["order_status"].isin(["canceled", "unavailable"])
].merge(importe_pedido, on="order_id", how="inner")

# 3. Fecha de referencia: el día siguiente a la última compra del dataset
fecha_ref = validos["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print("Fecha de referencia:", fecha_ref.date())

# 4. Tabla de clientes con R, F y M
rfm = validos.groupby("customer_unique_id").agg(
    ultima_compra=("order_purchase_timestamp", "max"),
    num_pedidos=("order_id", "count"),
    gasto_total=("importe", "sum"),
)
rfm["recencia"] = (fecha_ref - rfm["ultima_compra"]).dt.days
rfm["repite"] = rfm["num_pedidos"] > 1

print(f"Clientes: {len(rfm)}")
print()
print(rfm[["recencia", "num_pedidos", "gasto_total"]].describe().round(2))

Fecha de referencia: 2018-09-04
Clientes: 94989

       recencia  num_pedidos  gasto_total
count  94989.00     94989.00     94989.00
mean     243.44         1.03       165.69
std      152.99         0.21       226.74
min        1.00         1.00         9.59
25%      119.00         1.00        63.10
50%      224.00         1.00       107.90
75%      352.00         1.00       182.94
max      729.00        16.00     13664.08


Como casi todos los clientes compran una sola vez, la frecuencia no sirve para puntuar: los clientes que repiten forman el segmento **Fieles**, y el resto se clasifica por recencia y gasto usando cuartiles.

In [15]:
# 1. Puntuaciones del 1 al 4 por cuartiles
#    Recencia: menos días = mejor, por eso las etiquetas van al revés
rfm["R"] = pd.qcut(rfm["recencia"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["M"] = pd.qcut(rfm["gasto_total"], 4, labels=[1, 2, 3, 4]).astype(int)

# 2. Asignamos un segmento a cada cliente
def segmento(fila):
    if fila["repite"]:
        return "Fieles"
    if fila["R"] >= 3 and fila["M"] >= 3:
        return "Recientes de alto valor"
    if fila["R"] >= 3:
        return "Recientes de bajo valor"
    if fila["M"] >= 3:
        return "Alto valor en riesgo"
    return "Perdidos de bajo valor"

rfm["segmento"] = rfm.apply(segmento, axis=1)

# 3. Resumen por segmento
resumen_seg = rfm.groupby("segmento").agg(
    clientes=("recencia", "count"),
    recencia_media=("recencia", "mean"),
    gasto_medio=("gasto_total", "mean"),
    gasto_total=("gasto_total", "sum"),
)
resumen_seg["pct_clientes"] = resumen_seg["clientes"] / resumen_seg["clientes"].sum() * 100
resumen_seg["pct_ingresos"] = resumen_seg["gasto_total"] / resumen_seg["gasto_total"].sum() * 100

print(resumen_seg.sort_values("gasto_total", ascending=False).round(1))

                         clientes  recencia_media  gasto_medio  gasto_total  \
segmento                                                                      
Recientes de alto valor     22888           117.1        261.9    5993850.7   
Alto valor en riesgo        22068           370.4        265.5    5859798.3   
Perdidos de bajo valor      24026           370.7         63.9    1536063.6   
Recientes de bajo valor     23119           117.2         63.1    1458890.3   
Fieles                       2888           226.5        308.4     890534.1   

                         pct_clientes  pct_ingresos  
segmento                                             
Recientes de alto valor          24.1          38.1  
Alto valor en riesgo             23.2          37.2  
Perdidos de bajo valor           25.3           9.8  
Recientes de bajo valor          24.3           9.3  
Fieles                            3.0           5.7  


Los dos segmentos de alto valor concentran el **75 % de los ingresos**. El segmento *Alto valor en riesgo* (37 % de los ingresos) son clientes que gastaron mucho pero llevan, de media, más de un año sin comprar.

## 10. Exportación de datos para Power BI

Guardamos en `data/processed/` las tablas que alimentan el dashboard.

In [16]:
ruta_salida = "data/processed/"

# Pedidos entregados con retraso, reseña y estado del cliente (base de casi todo el dashboard)
pedidos_estado.to_csv(ruta_salida + "pedidos_entregados.csv", index=False)

# Pedidos por categoría
pedidos_cat.to_csv(ruta_salida + "pedidos_categoria.csv", index=False)

# Resumen por vendedor
resumen_vend.reset_index().to_csv(ruta_salida + "resumen_vendedores.csv", index=False)

# Clientes con su segmento
rfm.reset_index().to_csv(ruta_salida + "clientes_segmentos.csv", index=False)

print("Archivos guardados:")
import os
print(os.listdir(ruta_salida))

Archivos guardados:
['clientes_segmentos.csv', 'pedidos_categoria.csv', 'pedidos_entregados.csv', 'resumen_vendedores.csv']


## 11. Conclusiones

- **Los retrasos hunden la satisfacción:** la nota media baja de 4,29 a 2,27 cuando un pedido llega tarde.
- **El problema es geográfico:** los estados del norte y nordeste tienen hasta 5 veces más retrasos que São Paulo.
- **La categoría influye poco**, salvo en muebles de oficina.
- **Un grupo pequeño de vendedores** acumula porcentajes de retraso muy por encima de la media.
- **Casi nadie repite** (3,1 %), y un retraso en la primera compra reduce aún más la probabilidad de volver.
- **Oportunidad:** el segmento *Alto valor en riesgo* representa más de un tercio de los ingresos y sería el objetivo prioritario de campañas de reactivación.